# Fase 2 – Data Understanding
Esta fase segue as quatro tarefas da metodologia CRISP-DM. Em cada tarefa são aplicados os passos do processo EDA correspondentes:

| Tarefa CRISP-DM | Passos EDA |
|---|---|
| 2.1 Recolha inicial dos dados | Passos 1 e 2 – Compreender e inspecionar |
| 2.2 Descrição dos dados | Passos 4 e 6 – Explorar e visualizar |
| 2.3 Exploração dos dados | Passos 5 e 8 – Transformar e reportar |
| 2.4 Verificação da qualidade | Passos 3 e 7 – Ausentes e outliers |

Nesta fase os problemas são **identificados**; a sua correção é feita na Fase 3 – Data Preparation.

## 2.1 Recolha Inicial dos Dados (Collect Initial Data)
> **CRISP-DM:** obter os dados listados nos recursos do projeto e carregá-los na ferramenta de análise. Output: *Initial Data Collection Report*, com os datasets obtidos, a sua localização, o método de obtenção e os problemas encontrados, bem como a forma como foram resolvidos.
>
> **Passos EDA:** 1 – Compreensão do problema e dos dados; 2 – Importação e inspeção inicial.

**Contexto (do Business Understanding):** os dados provêm de uma UCIN e estão em duas tabelas ligadas pelo código de episódio. O episódio é um acontecimento clínico e não identifica o bebé. Os sinais vitais são horários e as patologias diárias.

In [1]:
# =============================================================================
# 2.1.1 Bibliotecas
# pandas: tabelas | numpy: cálculos | matplotlib/seaborn: gráficos
# os: verificar ficheiros na pasta de dados
# =============================================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [2]:
# =============================================================================
# 2.1.2 Localização dos dados
# Os ficheiros foram descarregados da competição Kaggle da UC e guardados na
# pasta "dados/". Lista o conteúdo da pasta e o tamanho de cada ficheiro,
# para documentar exatamente o que foi obtido.
# =============================================================================
PASTA_DADOS = "dados/"
FICH_SINAIS = "sinais_vitais_por_hora.csv"
FICH_PATOLOGIAS = "patologias_por_dia.csv"

for f in sorted(os.listdir(PASTA_DADOS)):
    tamanho_kb = os.path.getsize(os.path.join(PASTA_DADOS, f)) / 1024
    print(f"{f:45s} {tamanho_kb:8.1f} KB")

patologias_por_dia.csv                            28.1 KB
sinais_vitais_por_hora.csv                       367.2 KB


In [3]:
# =============================================================================
# 2.1.3 Inspeção do ficheiro em bruto
# Antes de usar o pandas, lê as primeiras linhas como texto. Assim confirmamos
# o separador (vírgula ou ponto e vírgula), o separador decimal e se há
# cabeçalho. Qualquer surpresa aqui é um "problema encontrado" a registar.
# =============================================================================
for fich in [FICH_SINAIS, FICH_PATOLOGIAS]:
    print(f"--- {fich} ---")
    with open(os.path.join(PASTA_DADOS, fich), encoding="utf-8", errors="replace") as f:
        for _ in range(5):
            print(f.readline().rstrip())
    print()

--- sinais_vitais_por_hora.csv ---
episodio,data,hora,sat_o2,puls_rate,perfusion
EP001,2024-01-30,9,93.5,158.63,0.99
EP001,2024-01-30,10,92.3,155.86,0.86
EP002,2024-01-30,10,94.31,133.2,0.91
EP001,2024-01-30,11,93.96,150.0,1.13

--- patologias_por_dia.csv ---
episodio,data,patologia
EP001,2024-01-30,Displasia broncopulmonar
EP001,2024-01-30,Refluxo gastroesofágico
EP003,2024-01-30,Doença das membranas hialinas
EP003,2024-01-30,Doença das membranas hialinas



In [4]:
# =============================================================================
# 2.1.4 Carregamento dos dados
# Tenta ler em UTF-8; se falhar (acentos mal codificados), usa latin-1.
# A codificação usada fica registada para o relatório de recolha.
# =============================================================================
def carregar_csv(nome):
    caminho = os.path.join(PASTA_DADOS, nome)
    try:
        df = pd.read_csv(caminho, encoding="utf-8")
        cod = "utf-8"
    except UnicodeDecodeError:
        df = pd.read_csv(caminho, encoding="latin-1")
        cod = "latin-1"
    print(f"{nome}: carregado com codificação {cod}")
    return df

df_sinais = carregar_csv(FICH_SINAIS)
df_patologias = carregar_csv(FICH_PATOLOGIAS)

sinais_vitais_por_hora.csv: carregado com codificação utf-8
patologias_por_dia.csv: carregado com codificação utf-8


In [5]:
# =============================================================================
# 2.1.5 Volumetria
# Nº de registos e campos de cada tabela. Confirma também os valores
# referidos no relatório de Business Understanding (9 857 e 696 registos).
# =============================================================================
for nome, df, esperado in [("Sinais vitais", df_sinais, 9857),
                           ("Patologias", df_patologias, 696)]:
    linhas, colunas = df.shape
    ok = "OK" if linhas == esperado else f"DIFERENTE do relatório ({esperado})"
    print(f"{nome}: {linhas} registos x {colunas} campos -> {ok}")

Sinais vitais: 9857 registos x 6 campos -> OK
Patologias: 696 registos x 3 campos -> OK


In [6]:
# =============================================================================
# 2.1.6 Campos obtidos vs. campos esperados
# Compara os nomes das colunas com o dicionário de dados. Apanha espaços
# escondidos, maiúsculas diferentes ou colunas a mais/em falta.
# =============================================================================
esperado = {
    "Sinais vitais": ["episodio", "data", "hora", "sat_o2", "puls_rate", "perfusion"],
    "Patologias": ["episodio", "data", "patologia"],
}
for nome, df in [("Sinais vitais", df_sinais), ("Patologias", df_patologias)]:
    obtidas = df.columns.tolist()
    print(f"{nome}: {obtidas}")
    print("   Em falta:", set(esperado[nome]) - set(obtidas) or "nenhuma")
    print("   A mais:  ", set(obtidas) - set(esperado[nome]) or "nenhuma")

Sinais vitais: ['episodio', 'data', 'hora', 'sat_o2', 'puls_rate', 'perfusion']
   Em falta: nenhuma
   A mais:   nenhuma
Patologias: ['episodio', 'data', 'patologia']
   Em falta: nenhuma
   A mais:   nenhuma


In [7]:
# =============================================================================
# 2.1.7 Tipos atribuídos e valores não nulos
# Esperado: hora -> int64 | sat_o2, puls_rate, perfusion -> float64
#           episodio, data, patologia -> object (texto)
# Uma coluna numérica lida como object indica valores não numéricos lá dentro
# (gralha a analisar na tarefa 2.4).
# =============================================================================
print("===== SINAIS VITAIS =====")
df_sinais.info()
print("\n===== PATOLOGIAS =====")
df_patologias.info()

===== SINAIS VITAIS =====
<class 'pandas.DataFrame'>
RangeIndex: 9857 entries, 0 to 9856
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   episodio   9857 non-null   str  
 1   data       9857 non-null   str  
 2   hora       9857 non-null   int64
 3   sat_o2     9813 non-null   str  
 4   puls_rate  9793 non-null   str  
 5   perfusion  9815 non-null   str  
dtypes: int64(1), str(5)
memory usage: 462.2 KB

===== PATOLOGIAS =====
<class 'pandas.DataFrame'>
RangeIndex: 696 entries, 0 to 695
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   episodio   696 non-null    str  
 1   data       696 non-null    str  
 2   patologia  684 non-null    str  
dtypes: str(3)
memory usage: 16.4 KB


In [8]:
# =============================================================================
# 2.1.8 Primeiras linhas e amostra aleatória
# head() mostra o início do ficheiro; sample() mostra linhas de todo o
# ficheiro, onde é mais provável aparecerem gralhas.
# random_state fixa a amostra para ser sempre a mesma.
# =============================================================================
display(df_sinais.head())
display(df_sinais.sample(8, random_state=42))
display(df_patologias.head())
display(df_patologias.sample(8, random_state=42))

,episodio,data,hora,sat_o2,puls_rate,perfusion
0,EP001,2024-01-30,9,93.5,158.63,0.99
1,EP001,2024-01-30,10,92.3,155.86,0.86
2,EP002,2024-01-30,10,94.31,133.2,0.91
3,EP001,2024-01-30,11,93.96,150.0,1.13
4,EP002,2024-01-30,11,91.19,139.79,1.55


,episodio,data,hora,sat_o2,puls_rate,perfusion
8376,EP026,2024-04-14,5,95.24,156.27,1.19
500,EP001,2024-02-04,19,92.82,162.51,1.1
8919,EP034,2024-04-16,18,98.0,145.38,1.46
872,EP001,2024-02-08,22,91.13,154.28,1.12
5414,EP022,2024-03-17,7,94.4,149.55,2.29
5558,EP025,2024-03-18,2,97.49,126.09,1.17
7185,EP026,2024-04-03,20,94.44,146.92,0.63
7048,EP026,2024-04-01,1,96.09,157.7,1.23


,episodio,data,patologia
0,EP001,2024-01-30,Displasia broncopulmonar
1,EP001,2024-01-30,Refluxo gastroesofágico
2,EP003,2024-01-30,Doença das membranas hialinas
3,EP003,2024-01-30,Doença das membranas hialinas
4,EP003,2024-01-30,Icterícia neonatal


,episodio,data,patologia
674,EP026,2024-04-19,Displasia broncopulmonar
314,EP016,2024-03-05,Doença das membranas hialinas
611,EP039,2024-04-13,Doença das membranas hialinas
431,EP027,2024-03-17,Citomegalovírus
552,EP027,2024-04-03,Citomegalovírus
211,EP015,2024-02-24,Doença das membranas hialinas
538,EP032,2024-03-30,Sépsis
176,EP001,2024-02-20,Displasia broncopulmonar


In [9]:
# =============================================================================
# 2.1.9 Ligação entre as duas fontes
# O CRISP-DM lembra que, com várias fontes, a integração é uma questão a
# considerar já aqui. Verifica se as tabelas falam dos mesmos episódios.
# (A integração propriamente dita é feita na Fase 3.)
# =============================================================================
ep_sinais = set(df_sinais["episodio"])
ep_patologias = set(df_patologias["episodio"])

print("Episódios nos sinais vitais:", len(ep_sinais))
print("Episódios nas patologias:   ", len(ep_patologias))
print("Em ambas as tabelas:        ", len(ep_sinais & ep_patologias))
print("Só nos sinais vitais:       ", len(ep_sinais - ep_patologias))
print("Só nas patologias:          ", len(ep_patologias - ep_sinais))

Episódios nos sinais vitais: 58
Episódios nas patologias:    45
Em ambas as tabelas:         40
Só nos sinais vitais:        18
Só nas patologias:           5


In [10]:
# =============================================================================
# 2.1.10 Episódios presentes só numa das tabelas
# Lista os códigos para documentar a discrepância. Se houver códigos
# parecidos (ex.: "EP01" vs "EP001", "ep001"), é gralha e não falta de dados.
# A análise completa é feita na tarefa 2.4.
# =============================================================================
print("Só nas patologias:   ", sorted(ep_patologias - ep_sinais))
print("Só nos sinais vitais:", sorted(ep_sinais - ep_patologias))

Só nas patologias:    [' EP009', ' EP020', 'EP 018', 'EP 037', 'ep029']
Só nos sinais vitais: [' EP011', ' EP016', ' EP027', 'EP 016', 'EP 026', 'EP 034', 'EP 036', 'EP007', 'EP012', 'EP021', 'EP028', 'EP031', 'EP035', 'EP044', 'EP047', 'ep016', 'ep019', 'ep023']


### Relatório de Recolha Inicial dos Dados

**Datasets obtidos**

| Dataset | Localização | Método de obtenção | Registos x campos | Formato |
|---|---|---|---|---|
| sinais_vitais_por_hora.csv | dados/ | Download da competição Kaggle da UC | 9 857 x 6 | CSV, separador vírgula, ponto decimal, UTF-8 |
| patologias_por_dia.csv | dados/ | Download da competição Kaggle da UC | 696 x 3 | CSV, separador vírgula, ponto decimal, UTF-8 |

Os dois ficheiros foram carregados em Python (pandas) sem erros de leitura. O número de registos coincide com o indicado no Business Understanding e todos os campos do dicionário de dados estão presentes, sem colunas adicionais. Não existem outras fontes de dados para o projeto.

**Problemas encontrados**

| Problema | Onde | Detalhe | Resolução |
|---|---|---|---|
| Colunas numéricas lidas como texto | sat_o2, puls_rate, perfusion | Existem valores não numéricos nestas colunas | Identificar os valores na tarefa 2.4; converter para número na Fase 3 |
| Valores em falta | sat_o2 (44), puls_rate (64), perfusion (42), patologia (12) | Representados como células vazias | Caracterizar na tarefa 2.4; definir tratamento na Fase 3 |
| Códigos de episódio inconsistentes | Ambas as tabelas | Espaço no início (" EP009"), espaço no meio ("EP 018") e minúsculas ("ep029") | Normalizar os códigos (remover espaços e converter para maiúsculas) na Fase 3 |

**Correspondência entre tabelas**

Sem normalização, os sinais vitais têm 58 códigos de episódio e as patologias 45, com 40 em comum. A maior parte da diferença deve-se às gralhas nos códigos, que fazem o mesmo episódio aparecer com vários códigos. Depois de normalizados, os episódios só com patologias deverão desaparecer. Ficam apenas com sinais vitais, e sem patologias registadas, oito episódios: EP007, EP012, EP021, EP028, EP031, EP035, EP044 e EP047. Estes números serão confirmados na tarefa 2.4.

**Nota sobre integração:** as duas fontes ligam-se pelos campos `episodio` e `data`, mas têm granularidades diferentes (sinais vitais por hora e patologias por dia). A integração será feita na Fase 3, depois de corrigidos os códigos de episódio e os restantes problemas de qualidade.

## 2.2 Descrição dos Dados (Describe Data)
> **CRISP-DM:** examinar as propriedades "de superfície" dos dados obtidos: formato, quantidade (registos e campos de cada tabela), identidade dos campos e outras características gerais. Avaliar se os dados satisfazem os requisitos do projeto. Output: *Data Description Report*.
>
> **Passos EDA:** 4 – Medidas de tendência central e dispersão; 6 – Visualização (análise univariada).

**Nota:** para calcular estatísticas é criada uma cópia de trabalho de cada tabela, com os valores numéricos convertidos e os códigos de episódio normalizados. Os valores que não é possível converter são excluídos das estatísticas e analisados na tarefa 2.4. Os dados originais não são alterados.